# Global unemployment

Unemployment rates across economies since 1991 (ILO modelled estimates via the World Bank), with the latest reading, the change on the prior year, and the gap between each economy and the world rate.

**Datasets:** `worldbank/unemployment_total`, `ilostat/unemployment_rate_modelled`

This notebook reads the latest vintage each time it runs; re-run after a new release to update every table and chart.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from labobs.load import load, vintage, curated, raw_payload
from labobs.nb import select, pivot, latest, plot_lines, plot_bars, to_annual, FOCUS, FOCUS_NAMES
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
DATASETS = ["worldbank/unemployment_total", "ilostat/unemployment_rate_modelled"]
for k in DATASETS:
    v = vintage(k)
    print(f"{k:50} vintage {v.get('vintage', 'not yet ingested'):12} sha {v.get('sha256', '')[:12]}")

## Latest reading by economy

In [ ]:
u = load("worldbank/unemployment_total")
latest(select(u, ref_area=FOCUS))

In [ ]:
p = pivot(select(u, ref_area=FOCUS))
plot_lines(p, "Unemployment rate, % of labour force", "%", "ILO modelled estimates via World Bank WDI (SL.UEM.TOTL.ZS)");

## Distance from the world rate

In [ ]:
gap = p.sub(p["WLD"], axis=0).drop(columns="WLD")
gap.tail(5).T.sort_values(gap.index[-1])

## By sex (ILO modelled estimates)
Rows are filtered to total age 15+; `sex` takes SEX_T, SEX_M, SEX_F.

In [ ]:
ilo = load("ilostat/unemployment_rate_modelled")
print(ilo.columns.tolist())
print(ilo["classif1"].drop_duplicates().head(20).tolist())

In [ ]:
f = select(ilo, ref_area=FOCUS, classif1="AGE_YTHADULT_YGE15")
sexgap = pivot(f, columns="sex").assign(gap=lambda d: d["SEX_F"] - d["SEX_M"])
sexgap.tail(10)

## Curator notes

_Written by the curator agent after each new vintage: what changed, which revisions matter, and what to watch. See `DATA_CHANGELOG.md` for the full log._